# Лабораторная работа 6

**Тема:** численное интегрирование и интегрирование статистическим методом Монте-Карло.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

**Вариант 6:** $f(x) = \dfrac{1}{1 + x^2}$, $\quad a = 0.5$, $\quad b = 2$

**Задание:**
1. Построить график подынтегральной функции на $[a, b]$.
2. Записать и вычислить аналитическое выражение $I = \int_a^b f(x)\,dx$.
3. Вычислить интеграл методом трапеций и определить число узлов сетки, обеспечивающее относительную ошибку 1%.
4. Вычислить интеграл $I^*$ методом Монте-Карло двумя способами с такой же относительной ошибкой 1%.
   Для каждого способа рассчитать выборочное СКО для выборки из оценок $\{I_i^*\}$, $i = 1, \dots, 100$.
5. Сравнить результаты всех вычислений.
6. При выводе показать результат, его точность и необходимое число итераций.

## Импорт библиотек и параметры

`NormalDist` из стандартной библиотеки нужна для значения $t$: это аргумент функции Лапласа
при заданной надёжности $\gamma$ (формулы (13), (14)).

In [ ]:
import math
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 100

In [ ]:
ASK_INPUT = False   # True — ввод параметров с клавиатуры при запуске
SEED = None         # None — от таймера; число — воспроизводимый результат


def ask(prompt, default, cast=float):
    if not ASK_INPUT:
        return default
    s = input(f"{prompt} [{default}]: ").strip()
    return cast(s) if s else default


a = ask("Нижний предел a", 0.5)
b = ask("Верхний предел b", 2.0)
EPS = ask("Допустимая относительная ошибка", 0.01)
GAMMA = ask("Надёжность γ для Монте-Карло", 0.997)
N0 = ask("Предварительное число испытаний N0", 1000, int)
RUNS = 100                                    # объём выборки оценок {I_i*}

t_gamma = NormalDist().inv_cdf((1 + GAMMA) / 2)   # 2Φ(t) = γ
rng = np.random.default_rng(SEED)


def f(x):
    return 1 / (1 + x ** 2)


print(f"f(x) = 1/(1 + x²),  [a, b] = [{a}, {b}]")
print(f"Допустимая ошибка ε = {EPS:.0%},  надёжность γ = {GAMMA},  t = {t_gamma:.3f}")

## 1. График подынтегральной функции

На $[0.5;\ 2]$ функция положительна и монотонно убывает от $f(0.5) = 0.8$ до $f(2) = 0.2$.
Площадь под кривой и есть искомый интеграл.

In [ ]:
xs = np.linspace(a, b, 500)

plt.figure(figsize=(10, 5))
plt.plot(xs, f(xs), "b", lw=2, label="f(x) = 1/(1 + x²)")
plt.fill_between(xs, f(xs), alpha=0.2, label="I = площадь под кривой")
plt.axhline(0, color="black", lw=0.8)
plt.title("Подынтегральная функция, вариант 6")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(alpha=0.4)
plt.legend()
plt.show()

## 2. Аналитическое решение

Первообразная табличная: $\int \dfrac{dx}{1 + x^2} = \operatorname{arctg} x + C$.
По формуле Ньютона–Лейбница (1):

$$I = \int_{0.5}^{2} \frac{dx}{1 + x^2} = \operatorname{arctg} 2 - \operatorname{arctg} 0.5.$$

По формуле разности арктангенсов $\operatorname{arctg}\alpha - \operatorname{arctg}\beta = \operatorname{arctg}\dfrac{\alpha - \beta}{1 + \alpha\beta}$:

$$I = \operatorname{arctg}\frac{2 - 0.5}{1 + 2 \cdot 0.5} = \operatorname{arctg}\frac{3}{4} \approx 0.6435011088.$$

In [ ]:
I_exact = math.atan(b) - math.atan(a)

print(f"I = arctg({b}) − arctg({a}) = {math.atan(b):.10f} − {math.atan(a):.10f} = {I_exact:.10f}")
print(f"Проверка: arctg(3/4) = {math.atan(0.75):.10f}")

## 3. Метод трапеций

На сетке $x_i = a + i h$, $h = (b - a)/N$, формула (6):

$$I \approx h\left[\frac{f(a) + f(b)}{2} + \sum_{i=1}^{N-1} f(x_i)\right].$$

Число разбиений $N$ увеличивается, пока относительная ошибка не станет меньше 1%. Число **узлов** сетки равно $N + 1$.

Результат проверяется двумя способами:
* **априорной оценкой** (7): $R \le \dfrac{(b - a)h^2}{12}\max|f''(\xi)|$, где $f''(x) = \dfrac{6x^2 - 2}{(1 + x^2)^3}$;
* **двойным счётом** с $N$ и $2N$ разбиениями. Разность $|I_N - I_{2N}|$ служит практической оценкой погрешности.

In [ ]:
def trapezoid(n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * ((y[0] + y[-1]) / 2 + np.sum(y[1:-1]))       # (6)


def f2(x):
    return (6 * x ** 2 - 2) / (1 + x ** 2) ** 3


history = []
n_trap = 1
while True:
    I_trap = trapezoid(n_trap)
    err_trap = abs(I_trap - I_exact) / abs(I_exact)
    history.append((n_trap, I_trap, err_trap))
    if err_trap <= EPS:
        break
    n_trap += 1

M2 = np.max(np.abs(f2(np.linspace(a, b, 100001))))
R_bound = (b - a) * ((b - a) / n_trap) ** 2 / 12 * M2       # (7)
I_2n = trapezoid(2 * n_trap)

print(f"{'N':>4}{'узлов':>7}{'I_N':>14}{'отн. ошибка, %':>17}")
for n, val, e in history:
    print(f"{n:>4}{n + 1:>7}{val:>14.8f}{e * 100:>17.4f}")

print(f"\nДостаточно N = {n_trap} разбиений, то есть {n_trap + 1} узлов сетки")
print(f"I_трап = {I_trap:.8f},  |I − I_трап| = {abs(I_trap - I_exact):.2e}  ({err_trap:.3%})")
print(f"Оценка (7): max|f''| = {M2:.4f},  R ≤ {R_bound:.2e}  ({R_bound / I_exact:.3%})")
print(f"Двойной счёт: I_2N = {I_2n:.8f},  |I_N − I_2N| = {abs(I_trap - I_2n):.2e}")

In [ ]:
ns = np.arange(1, 41)
errs = [abs(trapezoid(n) - I_exact) / I_exact * 100 for n in ns]

plt.figure(figsize=(10, 5))
plt.semilogy(ns, errs, "o-", ms=4, label="Относительная ошибка метода трапеций")
plt.axhline(EPS * 100, color="r", ls="--", label=f"Порог {EPS:.0%}")
plt.axvline(n_trap, color="g", ls=":", label=f"N = {n_trap}")
plt.title("Метод трапеций: ошибка убывает как h² ~ 1/N²")
plt.xlabel("Число разбиений N")
plt.ylabel("Относительная ошибка, %")
plt.grid(alpha=0.4, which="both")
plt.legend()
plt.show()

## 4. Метод Монте-Карло

### Способ 1 — выборочное среднее

$U$ равномерна на $[a, b]$, тогда $I = (b - a)\,M\big(f(U)\big)$ (9). Алгоритм:
$u_i = r_i(b - a) + a$, оценка по формуле (10):

$$I^* = \frac{b - a}{N}\sum_{i=1}^{N} f(u_i).$$

### Способ 2 — геометрический (попадание точек под кривую)

Кривая вписывается в прямоугольник $[a, b] \times [0, M]$, где $M = \max f = f(0.5) = 0.8$.
Бросаются точки $X = a + (b - a)r_1$, $Y = M r_2$. Считается число $k$ точек под кривой, оценка по формуле (11):

$$I^* = M\,(b - a)\,\frac{k}{N}.$$

### Как выбрать $N$ для ошибки 1%

По формулам (13), (14) с надёжностью $\gamma$ ошибка $|I^* - I| < \delta = t\,\sigma/\sqrt{N}$.
Требуем $\delta = \varepsilon\,I$:

$$N = \left(\frac{t\,\sigma}{\varepsilon\,I}\right)^2.$$

Здесь $\sigma$ — СКО одного испытания. Оно заранее неизвестно, поэтому действуем как в теории:
1. Проводим $N_0$ предварительных испытаний и оцениваем по ним $\sigma^*$.
2. Вычисляем $N$. Если $N > N_0$, проводим недостающие испытания.

Берём $\gamma = 0.997$, это правило трёх сигм ($t \approx 3$). Тогда ошибка **каждой** оценки
практически наверняка меньше 1%, а не только среднего из многих запусков.

In [ ]:
M_top = float(np.max(f(np.linspace(a, b, 100001))))     # высота прямоугольника


def mc_trials_mean(n):
    u = a + (b - a) * rng.random(n)                     # u_i = r_i(b − a) + a
    return (b - a) * f(u)                                # слагаемые (10)


def mc_trials_geom(n):
    x = a + (b - a) * rng.random(n)
    y = M_top * rng.random(n)
    return M_top * (b - a) * (y <= f(x))                 # слагаемые (11): S·[точка под кривой]


def required_n(trials, name):
    pilot = trials(N0)
    sigma = pilot.std(ddof=1)                            # σ* по N0 испытаниям
    n = math.ceil((t_gamma * sigma / (EPS * I_exact)) ** 2)
    print(f"{name}: σ* по N0 = {N0} испытаниям = {sigma:.4f}  →  N = (tσ*/(εI))² = {n}")
    if n > N0:
        print(f"   N > N0, нужны дополнительные испытания: {n - N0}")
    return n, sigma


N_mc1, sigma1 = required_n(mc_trials_mean, "Способ 1 (среднее)")
N_mc2, sigma2 = required_n(mc_trials_geom, "Способ 2 (геометрический)")

### Выборка из 100 оценок $\{I_i^*\}$

Для каждого способа 100 раз независимо вычисляем $I_i^*$ с найденным $N$. Затем считаем:
* среднее и выборочное СКО оценок $s = \sqrt{\frac{1}{99}\sum (I_i^* - \bar I^*)^2}$;
* теоретическое СКО $\sigma/\sqrt{N}$ для сравнения;
* долю оценок, попавших в коридор $\pm1\%$ от точного значения. Она должна быть около $\gamma$.

In [ ]:
def run_series(trials, n):
    return np.array([trials(n).mean() for _ in range(RUNS)])


est1 = run_series(mc_trials_mean, N_mc1)
est2 = run_series(mc_trials_geom, N_mc2)


def report(name, est, n, sigma):
    s = est.std(ddof=1)
    rel = np.abs(est - I_exact) / I_exact
    print(name)
    print("-" * 64)
    print(f"  Число испытаний в одной оценке N      = {n}")
    print(f"  Среднее из {RUNS} оценок                 = {est.mean():.6f}")
    print(f"  Выборочное СКО оценок s               = {s:.6f}")
    print(f"  Теоретическое СКО σ*/√N               = {sigma / math.sqrt(n):.6f}")
    print(f"  Точность δ = t·s                      = {t_gamma * s:.6f}  ({t_gamma * s / I_exact:.2%})")
    print(f"  Макс. ошибка одной оценки             = {rel.max():.3%}")
    print(f"  Оценок с ошибкой ≤ 1%                 = {np.sum(rel <= EPS)} из {RUNS}")
    print()
    return s


s1 = report("Монте-Карло, способ 1 (выборочное среднее)", est1, N_mc1, sigma1)
s2 = report("Монте-Карло, способ 2 (геометрический)", est2, N_mc2, sigma2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

n_show = 2000
x_pts = a + (b - a) * rng.random(n_show)
y_pts = M_top * rng.random(n_show)
under = y_pts <= f(x_pts)
ax1.scatter(x_pts[under], y_pts[under], s=4, c="tab:green", label="под кривой (k)")
ax1.scatter(x_pts[~under], y_pts[~under], s=4, c="tab:red", label="над кривой")
ax1.plot(xs, f(xs), "b", lw=2, label="f(x)")
ax1.plot([a, b, b, a, a], [0, 0, M_top, M_top, 0], "k--", lw=1)
ax1.set_title(f"Способ 2: {n_show} точек, I* = {M_top * (b - a) * under.mean():.4f}")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.grid(alpha=0.4)
ax1.legend(loc="upper right")

bins = np.linspace(I_exact * (1 - 1.5 * EPS), I_exact * (1 + 1.5 * EPS), 25)
ax2.hist(est1, bins=bins, alpha=0.6, label=f"Способ 1, N = {N_mc1}")
ax2.hist(est2, bins=bins, alpha=0.6, label=f"Способ 2, N = {N_mc2}")
ax2.axvline(I_exact, color="k", lw=2, label="Точное I")
ax2.axvspan(I_exact * (1 - EPS), I_exact * (1 + EPS), color="g", alpha=0.1, label="Коридор ±1%")
ax2.set_title(f"Распределение {RUNS} оценок I*")
ax2.set_xlabel("I*")
ax2.set_ylabel("Число оценок")
ax2.grid(alpha=0.4)
ax2.legend()

plt.tight_layout()
plt.show()

### Скорость сходимости Монте-Карло

Из (13) видно, что ошибка убывает как $1/\sqrt{N}$. Чтобы получить ещё один верный знак,
нужно увеличить $N$ в 100 раз. Проверим это: для нескольких $N$ посчитаем СКО по 100 оценкам.

In [ ]:
n_grid = np.array([100, 300, 1000, 3000, 10000, 30000])
sd1 = [run_series(mc_trials_mean, n).std(ddof=1) for n in n_grid]
sd2 = [run_series(mc_trials_geom, n).std(ddof=1) for n in n_grid]

plt.figure(figsize=(10, 5))
plt.loglog(n_grid, sd1, "o-", label="Способ 1: СКО 100 оценок")
plt.loglog(n_grid, sd2, "s-", label="Способ 2: СКО 100 оценок")
plt.loglog(n_grid, sigma1 / np.sqrt(n_grid), "--", color="tab:blue", alpha=0.6, label="σ₁/√N")
plt.loglog(n_grid, sigma2 / np.sqrt(n_grid), "--", color="tab:orange", alpha=0.6, label="σ₂/√N")
plt.axhline(EPS * I_exact / t_gamma, color="r", ls=":", label="СКО, нужное для 1% при γ = 0.997")
plt.title("Монте-Карло: СКО оценки убывает как 1/√N")
plt.xlabel("N")
plt.ylabel("СКО оценки I*")
plt.grid(alpha=0.4, which="both")
plt.legend()
plt.show()

## 5. Сравнение результатов

In [ ]:
rows = [
    ("Аналитически", I_exact, 0.0, "—", "—"),
    ("Метод трапеций", I_trap, abs(I_trap - I_exact) / I_exact, f"{n_trap + 1} узлов", "—"),
    ("Монте-Карло, способ 1", est1.mean(), abs(est1.mean() - I_exact) / I_exact, f"{N_mc1} точек", f"{s1:.6f}"),
    ("Монте-Карло, способ 2", est2.mean(), abs(est2.mean() - I_exact) / I_exact, f"{N_mc2} точек", f"{s2:.6f}"),
]

print(f"{'Метод':<24}{'I':>12}{'Ошибка, %':>12}{'Итераций':>16}{'СКО {I*}':>12}")
print("-" * 76)
for name, val, e, n, s in rows:
    print(f"{name:<24}{val:>12.6f}{e * 100:>12.4f}{n:>16}{s:>12}")

print(f"\nСпособ 2 требует в {N_mc2 / N_mc1:.1f} раза больше испытаний, чем способ 1:")
print(f"σ₂/σ₁ = {sigma2 / sigma1:.2f}, а N растёт как σ²")

## Вывод

1. **Аналитически** $I = \operatorname{arctg} 2 - \operatorname{arctg} 0.5 = \operatorname{arctg}(3/4) \approx 0.643501$.
2. **Метод трапеций** достигает ошибки 1% уже при нескольких узлах сетки. Функция гладкая,
   а ошибка метода убывает как $h^2$. Априорная оценка (7) и двойной счёт с $N$ и $2N$ согласуются
   с фактической ошибкой. Для одномерного интеграла от гладкой функции это самый экономичный способ.
3. **Метод Монте-Карло** требует тысяч и десятков тысяч испытаний. Его ошибка убывает всего
   как $1/\sqrt{N}$: чтобы уменьшить ошибку в 10 раз, нужно в 100 раз больше точек.
   Число испытаний выбрано по формулам (13), (14) с надёжностью $\gamma = 0.997$. Поэтому
   практически каждая из 100 оценок попадает в коридор $\pm1\%$, а выборочное СКО оценок
   совпадает с теоретическим $\sigma/\sqrt{N}$.
4. **Способ 1** (выборочное среднее) эффективнее **способа 2** (геометрического). В способе 2 каждое
   испытание несёт только один бит информации (попала точка под кривую или нет), поэтому
   дисперсия одного испытания больше и для той же точности нужно в несколько раз больше точек.
5. Преимущество Монте-Карло проявляется не здесь, а в многомерных задачах. Его трудоёмкость
   почти не зависит от размерности, тогда как для сеточных методов число узлов растёт как $N^k$.